In [1]:
%pip install sherpa-onnx soundfile numpy huggingface_hub librosa soundfile

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
os.environ.pop("SSL_CERT_FILE", None)

'C:\\Users\\Boss\\PycharmProjects\\Chess_AI\\Lib\\site-packages\\certifi\\cacert.pem'

In [ ]:
# files = [
#     'encoder-epoch-20-avg-10.int8.onnx',
#     'decoder-epoch-20-avg-10.int8.onnx',
#     'joiner-epoch-20-avg-10.int8.onnx',
#     'bpe.model',
#     'config.json'
# ]
# from huggingface_hub import hf_hub_download
# import os
# import traceback

# os.makedirs("models/stt", exist_ok=True)

# for f in files:
#     try:
#         path = hf_hub_download(
#             repo_id='hynt/Zipformer-30M-RNNT-6000h',
#             filename=f,
#             local_dir='./models/stt',
#             endpoint='https://hf-mirror.com'
#         )
#         print(f"Saved to: {path}")
#     except Exception as e:
#         traceback.print_exc() 

Saved to: models\stt\encoder-epoch-20-avg-10.int8.onnx
Saved to: models\stt\decoder-epoch-20-avg-10.int8.onnx
Saved to: models\stt\joiner-epoch-20-avg-10.int8.onnx
Saved to: models\stt\bpe.model


config.json: 0.00B [00:00, ?B/s]

Saved to: models\stt\config.json


In [ ]:
import soundfile as sf
import numpy as np
from sherpa_onnx import OfflineRecognizer

recognizer = OfflineRecognizer.from_transducer(
    encoder="models/stt/encoder-epoch-20-avg-10.int8.onnx",
    decoder="models/stt/decoder-epoch-20-avg-10.int8.onnx",
    joiner="models/stt/joiner-epoch-20-avg-10.int8.onnx",
    tokens="models/stt/config.json",      
    modeling_unit="bpe",                   
    bpe_vocab="models/stt/bpe.model",    
    num_threads=4,
)


In [8]:
# Load a wav file (16kHz mono)
audio, sr = sf.read("wavs/0.wav", dtype="float32")

In [10]:
# Transcribe
stream = recognizer.create_stream()
stream.accept_waveform(sr, audio)
recognizer.decode_stream(stream)

print("Result:", stream.result.text)

Result:  TÔN KÍNH THẦN LINH HAY LÀ HỌ LÀ NHỮNG CON NGƯỜI SỐNG CÔ QUẠNH BIỆT LẬP HOANG DÃ NHƯ NHỮNG BẦY THÚ Ở TRONG RỪNG CHÚNG TA CHẲNG NÊN BỎ QUA MÀ KHÔNG ĐẾN THĂM HỎI TÌM HIỂU


In [25]:
import difflib


def compare(expected, actual):
    expected_words = expected.strip().lower().split()
    actual_words = actual.strip().lower().split()
    
    matcher = difflib.SequenceMatcher(None, expected_words, actual_words)
    
    results = []
    for opcode, i1, i2, j1, j2 in matcher.get_opcodes():
        #True, False
        status = True if opcode == "equal" else False
        for word in expected_words[i1:i2]:
            results.append((word, status))
    # return xem tu nao dung + score
    correct = sum( [ 1 if status else 0 for _, status in results ])
    score = correct / len(expected_words) if expected_words else 0
    return results, score
# example
expected = "xin chào em"
actual = "xin cao em"

results, score = compare(expected, actual)
print(results)
print(score)

[('xin', True), ('chào', False), ('em', True)]
0.6666666666666666
